## 1️⃣ Configuration

In [1]:
# ============================================================
# CONFIGURATION aktuell
# ============================================================

# Paths
DATA_DIR = '/project/data'
MODEL_DIR = '/project/models'
IMAGE_DIR = f'{DATA_DIR}/collection_images'  # Images mounted here

# Model parameters
EMBEDDING_DIM = 1280  # MobileNetV2 output
IMG_SIZE = 224
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# Recommendation parameters
TOP_K = 10  # Number of recommendations to return
SIMILARITY_METHOD = 'cosine'  # cosine, euclidean

# Scoring weights
VISUAL_SIMILARITY_WEIGHT = 0.70
CO_PURCHASE_WEIGHT = 0.30

# Filtering options (optional)
SAME_COLLECTION_BOOST = 1.2  # Boost score if from same collection
PRICE_RANGE_TOLERANCE = 0.5  # +/- 50% of cart average price
EXCLUDE_CART_ITEMS = True  # Don't recommend items already in cart

print(f"⚙️  Configuration loaded")
print(f"  Device: {DEVICE}")
print(f"  Top-K: {TOP_K}")
print(f"  Image directory: {IMAGE_DIR}")
print(f"  Weights: Visual={VISUAL_SIMILARITY_WEIGHT}, Co-purchase={CO_PURCHASE_WEIGHT}")

⚙️  Configuration loaded
  Device: cpu
  Top-K: 10
  Image directory: /project/data/collection_images
  Weights: Visual=0.7, Co-purchase=0.3


## 2️⃣ Import Libraries

In [2]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import json
from collections import defaultdict, Counter
from tqdm import tqdm

# PyTorch and vision
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# Scikit-learn
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Visualization
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Settings
pd.set_option('display.max_columns', None)
np.random.seed(42)
torch.manual_seed(42)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 3️⃣ Load Data & Pre-computed Embeddings

In [3]:
# ============================================================
# LOAD DATA SOURCES
# ============================================================

print("📂 Loading data sources...")

# 1. Product catalog (tab-delimited, not semicolon)
products = pd.read_csv(
    f'{DATA_DIR}/feed_a62656-2_de.csv', 
    delimiter='\t',  # Tab-delimited
    encoding='utf-8'
)
# Map 'id' column to 'artikel_id' if needed
if 'id' in products.columns and 'artikel_id' not in products.columns:
    products['artikel_id'] = products['id']
products['artikel_id'] = products['artikel_id'].astype(str)
print(f"  ✓ Loaded {len(products):,} products from catalog")

# 2. Interaction data (for co-purchase patterns)
interactions = pd.read_csv(f'{DATA_DIR}/sample_interactions.csv')
interactions['timestamp'] = pd.to_datetime(interactions['timestamp'])
interactions = interactions.dropna(subset=['product_id'])
interactions['product_id'] = interactions['product_id'].astype(int).astype(str)

# 3. Load pre-computed embeddings (if available)
embeddings_file = Path(MODEL_DIR) / "hybrid_product_embeddings.npz"

if embeddings_file.exists():
    print("\n📦 Loading pre-computed embeddings...")
    data = np.load(embeddings_file)
    product_ids = data['product_ids']
    embeddings = data['embeddings']
    
    # Create lookup dictionary
    product_embeddings = {pid: emb for pid, emb in zip(product_ids, embeddings)}
    print(f"  ✓ Loaded {len(product_embeddings):,} product embeddings")
else:
    print("\n⚠️  No pre-computed embeddings found. Will extract on-demand.")
    product_embeddings = {}

print(f"\n✓ Data loaded successfully:")
print(f"  Products in catalog: {len(products):,}")
print(f"  Products with embeddings: {len(product_embeddings):,}")
print(f"  Interactions: {len(interactions):,}")
print(f"  Date range: {interactions['timestamp'].min()} to {interactions['timestamp'].max()}")

📂 Loading data sources...
  ✓ Loaded 11,969 products from catalog

📦 Loading pre-computed embeddings...
  ✓ Loaded 966 product embeddings

✓ Data loaded successfully:
  Products in catalog: 11,969
  Products with embeddings: 966
  Interactions: 9,445
  Date range: 2025-11-13 23:00:08.819173+00:00 to 2025-11-14 22:59:55.634203+00:00


## 4️⃣ Image Embedding Extractor (On-Demand)

In [4]:
# ============================================================
# IMAGE EMBEDDING MODEL (for new products not in cache)
# ============================================================

class ImageEmbeddingExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = nn.Sequential(*list(mobilenet.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d(1)
        
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x

# Initialize model
device = torch.device(DEVICE)
embedding_model = ImageEmbeddingExtractor().to(device)
embedding_model.eval()

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_embedding(product_id):
    """
    Extract embedding for a product image.
    First checks cache, then extracts from image file.
    """
    product_id = str(product_id)
    
    # Check cache
    if product_id in product_embeddings:
        return product_embeddings[product_id]
    
    # Extract from image
    image_path = Path(IMAGE_DIR) / f"{product_id}.jpg"
    if not image_path.exists():
        print(f"⚠️  Image not found for product {product_id}")
        return None
    
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = embedding_model(img_tensor)
        
        embedding = embedding.cpu().numpy().flatten()
        
        # Cache for future use
        product_embeddings[product_id] = embedding
        return embedding
    except Exception as e:
        print(f"❌ Error extracting embedding for {product_id}: {e}")
        return None

print(f"✓ Image embedding model loaded on {DEVICE}")
print(f"  Model ready for on-demand extraction")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/workbench/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:01<00:00, 9.25MB/s]


✓ Image embedding model loaded on cpu
  Model ready for on-demand extraction


## 4½️⃣ Fine-tune on Jewelry (Recommended - Run Once)

**Why?** MobileNetV2 was trained on general images, not jewelry. Fine-tuning teaches it jewelry-specific features like metal types, stone settings, and design styles for better recommendations.

In [5]:
# ============================================================
# FINE-TUNE ON JEWELRY IMAGES (Self-Supervised)
# ============================================================

ENABLE_FINETUNING = True  # Set False to skip
EPOCHS = 5
LR = 0.0001

if not ENABLE_FINETUNING:
    print("ℹ️  Fine-tuning disabled")
else:
    print("🎨 Fine-tuning MobileNetV2 on jewelry images...\n")
    
    # Use product categories as pseudo-labels for self-supervised learning
    img_products = products[products['artikel_id'].isin([f.stem for f in Path(IMAGE_DIR).glob('*.jpg')])].copy()
    
    # Find category column
    cat_col = 'product_type' if 'product_type' in img_products.columns else 'google_product_category' if 'google_product_category' in img_products.columns else None
    
    if not cat_col or img_products[cat_col].isna().all():
        print("⚠️  No categories found - skipping")
        ENABLE_FINETUNING = False
    else:
        img_products = img_products.dropna(subset=[cat_col])
        img_products['cat'] = img_products[cat_col].astype(str).str.split('>').str[0].str.strip()
        
        # Keep categories with ≥10 products
        cat_counts = img_products['cat'].value_counts()
        valid_cats = cat_counts[cat_counts >= 10].index
        img_products = img_products[img_products['cat'].isin(valid_cats)]
        
        if len(valid_cats) < 2:
            print("⚠️  Not enough data - need ≥2 categories with ≥10 products each")
            ENABLE_FINETUNING = False
        else:
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            img_products['label'] = le.fit_transform(img_products['cat'])
            n_classes = len(valid_cats)
            
            print(f"  Categories: {n_classes}")
            for c, cnt in cat_counts[valid_cats].head(5).items():
                print(f"    • {c}: {cnt}")
            
            # Add classifier head
            class JewelryClassifier(nn.Module):
                def __init__(self, base, n_cls):
                    super().__init__()
                    self.base = base
                    self.fc = nn.Linear(EMBEDDING_DIM, n_cls)
                def forward(self, x):
                    emb = self.base.features(x)
                    emb = self.base.pool(emb).flatten(1)
                    return self.fc(emb), emb
            
            model = JewelryClassifier(embedding_model, n_classes).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=LR)
            
            # Simple dataset
            from torch.utils.data import Dataset, DataLoader
            class JData(Dataset):
                def __init__(self, df):
                    self.df = df.reset_index(drop=True)
                def __len__(self):
                    return len(self.df)
                def __getitem__(self, i):
                    r = self.df.iloc[i]
                    try:
                        img = Image.open(Path(IMAGE_DIR)/f"{r['artikel_id']}.jpg").convert('RGB')
                        return transform(img), int(r['label'])
                    except:
                        return torch.randn(3,IMG_SIZE,IMG_SIZE), 0
            
            loader = DataLoader(JData(img_products), batch_size=32, shuffle=True, num_workers=2)
            
            # Train
            print(f"\n🔧 Training {EPOCHS} epochs...")
            model.train()
            for e in range(EPOCHS):
                loss_sum, correct, total = 0, 0, 0
                for imgs, lbls in tqdm(loader, desc=f"Epoch {e+1}"):
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    optimizer.zero_grad()
                    out, _ = model(imgs)
                    loss = criterion(out, lbls)
                    loss.backward()
                    optimizer.step()
                    loss_sum += loss.item()
                    correct += (out.argmax(1) == lbls).sum().item()
                    total += len(lbls)
                acc = 100*correct/total
                print(f"  Epoch {e+1}: Loss={loss_sum/len(loader):.4f}, Acc={acc:.1f}%")
            
            # Update embedding model
            embedding_model = model.base
            embedding_model.eval()
            product_embeddings.clear()  # Re-extract needed
            
            # Save
            torch.save(embedding_model.state_dict(), Path(MODEL_DIR)/"jewelry_finetuned.pth")
            print(f"\n✅ Fine-tuned! Acc={acc:.1f}% | 💾 Saved | ⚠️  Re-extract embeddings next")

🎨 Fine-tuning MobileNetV2 on jewelry images...

  Categories: 7
    • Ohrschmuck: 436
    • Halsschmuck: 156
    • Fingerringe: 149
    • Armschmuck: 119
    • Anhänger: 67

🔧 Training 5 epochs...


Epoch 1: 100%|██████████| 31/31 [02:26<00:00,  4.72s/it]


  Epoch 1: Loss=0.6857, Acc=79.7%


Epoch 2: 100%|██████████| 31/31 [02:25<00:00,  4.69s/it]


  Epoch 2: Loss=0.1335, Acc=96.9%


...

  Epoch 3: Loss=0.0612, Acc=99.6%


Epoch 4: 100%|██████████| 31/31 [02:23<00:00,  4.62s/it]


  Epoch 4: Loss=0.0325, Acc=99.7%


Epoch 5: 100%|██████████| 31/31 [02:22<00:00,  4.58s/it]


  Epoch 5: Loss=0.0122, Acc=100.0%

✅ Fine-tuned! Acc=100.0% | 💾 Saved | ⚠️  Re-extract embeddings next


## 4¾️⃣ Extract Embeddings for All Products (Run Once)

In [6]:
# ============================================================
# EXTRACT EMBEDDINGS FOR ALL AVAILABLE PRODUCT IMAGES
# ============================================================

# Only extract if we don't have many embeddings yet
if len(product_embeddings) < 100:
    print("🖼️  Extracting embeddings for all product images...")
    print(f"  Starting with {len(product_embeddings):,} cached embeddings\n")
    
    # Get all image files
    image_files = list(Path(IMAGE_DIR).glob('*.jpg'))
    print(f"  Found {len(image_files):,} product images")
    
    # Extract embeddings for all products with images
    extracted_count = 0
    skipped_count = 0
    
    for img_file in tqdm(image_files, desc="Extracting embeddings"):
        product_id = img_file.stem
        
        # Skip if already in cache
        if product_id in product_embeddings:
            skipped_count += 1
            continue
        
        # Extract embedding
        emb = extract_embedding(product_id)
        if emb is not None:
            extracted_count += 1
    
    print(f"\n✅ Embedding extraction complete:")
    print(f"  Total embeddings: {len(product_embeddings):,}")
    print(f"  Newly extracted: {extracted_count:,}")
    print(f"  Already cached: {skipped_count:,}")
    
    # Save updated embeddings
    if extracted_count > 0:
        embeddings_file = Path(MODEL_DIR) / "hybrid_product_embeddings.npz"
        np.savez_compressed(
            embeddings_file,
            product_ids=list(product_embeddings.keys()),
            embeddings=np.array(list(product_embeddings.values()))
        )
        print(f"  💾 Saved to: {embeddings_file}")
else:
    print(f"✓ Already have {len(product_embeddings):,} embeddings - skipping extraction")

🖼️  Extracting embeddings for all product images...
  Starting with 0 cached embeddings

  Found 966 product images


Extracting embeddings: 100%|██████████| 966/966 [00:55<00:00, 17.52it/s]



✅ Embedding extraction complete:
  Total embeddings: 966
  Newly extracted: 966
  Already cached: 0
  💾 Saved to: /project/models/hybrid_product_embeddings.npz


## 5️⃣ Build Co-Purchase Matrix

In [7]:
# ============================================================
# CO-PURCHASE PATTERNS
# ============================================================

print("🔗 Building co-purchase patterns from interaction data...")

# Find products purchased together (same customer + purchase event)
purchases = interactions[interactions['interaction_type'] == 'purchase'].copy()

# Group by customer to find co-purchases
co_purchase_counts = defaultdict(lambda: defaultdict(int))

for customer_id, group in purchases.groupby('customer_id'):
    products_purchased = group['product_id'].unique()
    
    # Count co-occurrences
    for i, prod1 in enumerate(products_purchased):
        for prod2 in products_purchased[i+1:]:
            co_purchase_counts[prod1][prod2] += 1
            co_purchase_counts[prod2][prod1] += 1  # Symmetric

# Convert to dictionary
co_purchase_matrix = {k: dict(v) for k, v in co_purchase_counts.items()}

total_pairs = sum(len(v) for v in co_purchase_matrix.values())
print(f"\n✓ Co-purchase matrix built:")
print(f"  Products with co-purchase data: {len(co_purchase_matrix):,}")
print(f"  Total co-purchase pairs: {total_pairs:,}")
print(f"  Avg co-purchases per product: {total_pairs/len(co_purchase_matrix) if co_purchase_matrix else 0:.1f}")

🔗 Building co-purchase patterns from interaction data...

✓ Co-purchase matrix built:
  Products with co-purchase data: 0
  Total co-purchase pairs: 0
  Avg co-purchases per product: 0.0


## 6️⃣ Cart-Based Recommendation Engine

In [8]:
# ============================================================
# CART RECOMMENDATION FUNCTION
# ============================================================

def recommend_for_cart(cart_product_ids, top_k=TOP_K, verbose=True):
    """
    Generate product recommendations based on items in shopping cart.
    
    Args:
        cart_product_ids: List of product IDs currently in cart
        top_k: Number of recommendations to return
        verbose: Print progress and explanations
    
    Returns:
        DataFrame with recommended products and scores
    """
    if verbose:
        print(f"\n🛒 Generating recommendations for cart with {len(cart_product_ids)} items...")
    
    # Convert to strings
    cart_product_ids = [str(pid) for pid in cart_product_ids]
    
    # ==================== STEP 1: Extract cart embeddings ====================
    cart_embeddings = []
    valid_cart_ids = []
    
    for pid in cart_product_ids:
        emb = extract_embedding(pid)
        if emb is not None:
            cart_embeddings.append(emb)
            valid_cart_ids.append(pid)
    
    if not cart_embeddings:
        print("❌ No valid embeddings found for cart items")
        return pd.DataFrame()
    
    # Average cart embedding (represents overall style)
    cart_embedding_avg = np.mean(cart_embeddings, axis=0)
    
    if verbose:
        print(f"  ✓ Extracted embeddings for {len(valid_cart_ids)} cart items")
    
    # ==================== STEP 2: Calculate visual similarity ====================
    all_product_ids = list(product_embeddings.keys())
    all_embeddings = np.array([product_embeddings[pid] for pid in all_product_ids])
    
    # Compute similarity
    if SIMILARITY_METHOD == 'cosine':
        similarities = cosine_similarity([cart_embedding_avg], all_embeddings)[0]
    else:
        distances = euclidean_distances([cart_embedding_avg], all_embeddings)[0]
        similarities = 1 / (1 + distances)  # Convert to similarity
    
    # Create scores dictionary
    visual_scores = {pid: sim for pid, sim in zip(all_product_ids, similarities)}
    
    if verbose:
        print(f"  ✓ Calculated visual similarity for {len(all_product_ids):,} products")
    
    # ==================== STEP 3: Add co-purchase boost ====================
    co_purchase_scores = defaultdict(float)
    
    for cart_pid in valid_cart_ids:
        if cart_pid in co_purchase_matrix:
            for co_pid, count in co_purchase_matrix[cart_pid].items():
                # Normalize by total purchases
                co_purchase_scores[co_pid] += count
    
    # Normalize co-purchase scores to 0-1 range
    if co_purchase_scores:
        max_co_purchase = max(co_purchase_scores.values())
        co_purchase_scores = {k: v/max_co_purchase for k, v in co_purchase_scores.items()}
    
    if verbose:
        print(f"  ✓ Found co-purchase patterns for {len(co_purchase_scores)} products")
    
    # ==================== STEP 4: Combine scores ====================
    final_scores = {}
    
    for pid in all_product_ids:
        visual_score = visual_scores.get(pid, 0)
        co_purchase_score = co_purchase_scores.get(pid, 0)
        
        # Weighted combination
        final_score = (
            VISUAL_SIMILARITY_WEIGHT * visual_score +
            CO_PURCHASE_WEIGHT * co_purchase_score
        )
        
        final_scores[pid] = {
            'score': final_score,
            'visual_score': visual_score,
            'co_purchase_score': co_purchase_score
        }
    
    # ==================== STEP 5: Apply filters ====================
    # Get cart product info
    cart_products = products[products['artikel_id'].isin(valid_cart_ids)]
    
    # Calculate average cart price (if available)
    avg_cart_price = None
    if 'preis_eur' in cart_products.columns:
        cart_prices = pd.to_numeric(cart_products['preis_eur'], errors='coerce')
        if not cart_prices.isna().all():
            avg_cart_price = cart_prices.mean()
    
    # Get cart collections
    cart_collections = set()
    if 'kollektion_de' in cart_products.columns:
        cart_collections = set(cart_products['kollektion_de'].dropna().unique())
    
    # Apply filters and boosts
    for pid, score_dict in final_scores.items():
        # Exclude cart items
        if EXCLUDE_CART_ITEMS and pid in valid_cart_ids:
            score_dict['score'] = -1  # Mark for removal
            continue
        
        # Same collection boost
        if cart_collections:
            prod_info = products[products['artikel_id'] == pid]
            if not prod_info.empty and 'kollektion_de' in prod_info.columns:
                prod_collection = prod_info['kollektion_de'].values[0]
                if prod_collection in cart_collections:
                    score_dict['score'] *= SAME_COLLECTION_BOOST
        
        # Price range filter (optional)
        if avg_cart_price is not None and PRICE_RANGE_TOLERANCE > 0:
            prod_info = products[products['artikel_id'] == pid]
            if not prod_info.empty and 'preis_eur' in prod_info.columns:
                prod_price = pd.to_numeric(prod_info['preis_eur'].values[0], errors='coerce')
                if not pd.isna(prod_price):
                    price_ratio = prod_price / avg_cart_price
                    if price_ratio < (1 - PRICE_RANGE_TOLERANCE) or price_ratio > (1 + PRICE_RANGE_TOLERANCE):
                        score_dict['score'] *= 0.7  # Penalize out-of-range prices
    
    # ==================== STEP 6: Rank and return ====================
    # Remove excluded items
    final_scores = {k: v for k, v in final_scores.items() if v['score'] >= 0}
    
    # Sort by final score
    sorted_products = sorted(final_scores.items(), key=lambda x: x[1]['score'], reverse=True)
    
    # Get top K
    top_products = sorted_products[:top_k]
    
    # Create result DataFrame
    results = []
    for pid, scores in top_products:
        prod_info = products[products['artikel_id'] == pid]
        if not prod_info.empty:
            result = {
                'product_id': pid,
                'final_score': scores['score'],
                'visual_similarity': scores['visual_score'],
                'co_purchase_score': scores['co_purchase_score'],
            }
            
            # Add product metadata
            if 'bezeichnung_de' in prod_info.columns:
                result['product_name'] = prod_info['bezeichnung_de'].values[0]
            if 'preis_eur' in prod_info.columns:
                result['price'] = prod_info['preis_eur'].values[0]
            if 'kollektion_de' in prod_info.columns:
                result['collection'] = prod_info['kollektion_de'].values[0]
            if 'produktkategorie' in prod_info.columns:
                result['category'] = prod_info['produktkategorie'].values[0]
            
            results.append(result)
    
    recommendations_df = pd.DataFrame(results)
    
    if verbose:
        print(f"\n✅ Generated {len(recommendations_df)} recommendations")
        if not recommendations_df.empty:
            print(f"  Score range: {recommendations_df['final_score'].min():.3f} - {recommendations_df['final_score'].max():.3f}")
            print(f"  Avg visual similarity: {recommendations_df['visual_similarity'].mean():.3f}")
            if recommendations_df['co_purchase_score'].sum() > 0:
                print(f"  {(recommendations_df['co_purchase_score'] > 0).sum()} products with co-purchase history")
    
    return recommendations_df

print("✓ Cart recommendation engine ready")

✓ Cart recommendation engine ready


## 7️⃣ Test with Sample Cart

In [11]:
# ============================================================
# TEST RECOMMENDATIONS
# ============================================================

print("🧪 Testing cart recommendations...\n")

# ============================================================
# CONFIGURATION: Specify custom product IDs (or set to None for automatic selection)
# ============================================================
CUSTOM_CART_PRODUCT_IDS = ['598354']  # Example: ['598354']

# Get products that have both images and interactions
image_files = list(Path(IMAGE_DIR).glob('*.jpg'))
available_product_ids = [f.stem for f in image_files]

# Use custom product IDs if provided, otherwise auto-select
if CUSTOM_CART_PRODUCT_IDS is not None:
    # Validate that the specified products exist
    sample_cart = []
    for pid in CUSTOM_CART_PRODUCT_IDS:
        pid_str = str(pid)
        if pid_str in available_product_ids:
            sample_cart.append(pid_str)
        else:
            print(f"⚠️  Warning: Product ID '{pid}' not found in available products (no image). Skipping...")
    
    if len(sample_cart) == 0:
        print("❌ None of the specified product IDs are available. Falling back to automatic selection.")
        CUSTOM_CART_PRODUCT_IDS = None  # Trigger automatic selection below
    else:
        print(f"✅ Using {len(sample_cart)} custom product IDs from your selection")
else:
    # Automatic selection logic
    # Find interactions for products with images
    interactions_with_images = interactions[interactions['product_id'].isin(available_product_ids)]

    if len(interactions_with_images) > 0:
        # Get a sample cart from products that have images
        sample_customer = interactions_with_images[interactions_with_images['interaction_type'] == 'add_to_cart'].sample(1)
        sample_cart = interactions_with_images[
            (interactions_with_images['customer_id'] == sample_customer['customer_id'].values[0]) &
            (interactions_with_images['interaction_type'] == 'add_to_cart')
        ]['product_id'].unique()[:3].tolist()
        
        # If cart is empty or too small, use random products with images
        if len(sample_cart) == 0:
            sample_cart = np.random.choice(available_product_ids, size=min(3, len(available_product_ids)), replace=False).tolist()
    else:
        # Fallback: use any products with images
        sample_cart = available_product_ids[:3]

print(f"\n📦 Sample cart with {len(sample_cart)} items:")
for pid in sample_cart:
    prod = products[products['artikel_id'] == str(pid)]
    if not prod.empty and 'bezeichnung_de' in prod.columns:
        print(f"  • {pid}: {prod['bezeichnung_de'].values[0]}")
    else:
        print(f"  • {pid}")

# Generate recommendations
recommendations = recommend_for_cart(sample_cart, top_k=10, verbose=True)

# Display results
if not recommendations.empty:
    print("\n📋 Top Recommendations:")
    print("="*100)
    
    display_cols = ['product_id', 'final_score', 'visual_similarity', 'co_purchase_score']
    if 'product_name' in recommendations.columns:
        display_cols.insert(1, 'product_name')
    if 'price' in recommendations.columns:
        display_cols.insert(2, 'price')
    if 'collection' in recommendations.columns:
        display_cols.insert(3, 'collection')
    
    display(recommendations[display_cols].head(10))

🧪 Testing cart recommendations...

✅ Using 1 custom product IDs from your selection

📦 Sample cart with 1 items:
  • 598354

🛒 Generating recommendations for cart with 1 items...
  ✓ Extracted embeddings for 1 cart items
  ✓ Calculated visual similarity for 966 products
  ✓ Found co-purchase patterns for 0 products

✅ Generated 10 recommendations
  Score range: 0.658 - 0.665
  Avg visual similarity: 0.944

📋 Top Recommendations:


,product_id,final_score,visual_similarity,co_purchase_score
0,578287,0.665053,0.950076,0
1,557178,0.662986,0.947123,0
2,567669,0.662238,0.946054,0
3,574986,0.660797,0.943996,0
4,578347,0.659516,0.942165,0
5,585571,0.659383,0.941976,0
6,568092,0.659159,0.941655,0
7,575444,0.658911,0.941302,0
8,585572,0.658664,0.940949,0
9,574985,0.658127,0.940182,0


## 8️⃣ Visualize Recommendations

In [ ]:
# ============================================================
# VISUALIZE RECOMMENDATIONS
# ============================================================

def visualize_recommendations(cart_product_ids, recommendations_df, max_display=6):
    """
    Visualize cart items and recommendations with images.
    """
    fig, axes = plt.subplots(2, max_display, figsize=(20, 8))
    
    # Row 1: Cart items
    for idx, pid in enumerate(cart_product_ids[:max_display]):
        img_path = Path(IMAGE_DIR) / f"{pid}.jpg"
        if img_path.exists():
            img = Image.open(img_path)
            axes[0, idx].imshow(img)
            axes[0, idx].set_title(f"Cart: {pid}", fontsize=10, fontweight='bold')
        else:
            axes[0, idx].text(0.5, 0.5, f"No image\n{pid}", ha='center', va='center')
        axes[0, idx].axis('off')
    
    # Row 2: Recommendations
    for idx, row in recommendations_df.head(max_display).iterrows():
        if idx >= max_display:
            break
        
        pid = row['product_id']
        img_path = Path(IMAGE_DIR) / f"{pid}.jpg"
        if img_path.exists():
            img = Image.open(img_path)
            axes[1, idx].imshow(img)
            title = f"#{idx+1}: {pid}\nScore: {row['final_score']:.3f}"
            axes[1, idx].set_title(title, fontsize=10)
        else:
            axes[1, idx].text(0.5, 0.5, f"No image\n{pid}", ha='center', va='center')
        axes[1, idx].axis('off')
    
    plt.suptitle("🛒 Cart Items (top) → 🎯 Recommendations (bottom)", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

# Visualize the test results
if not recommendations.empty:
    visualize_recommendations(sample_cart, recommendations, max_display=6)

/tmp/ipykernel_7833/2765580277.py:39: UserWarning: Glyph 128722 (\N{SHOPPING TROLLEY}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_7833/2765580277.py:39: UserWarning: Glyph 127919 (\N{DIRECT HIT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()


## 9️⃣ Production API Function

In [ ]:
# ============================================================
# PRODUCTION-READY API FUNCTION
# ============================================================

def get_cart_recommendations_api(cart_product_ids, top_k=10):
    """
    Production API function for cart recommendations.
    Returns JSON-serializable result.
    
    Args:
        cart_product_ids: List of product IDs in cart
        top_k: Number of recommendations
    
    Returns:
        dict with recommendations and metadata
    """
    try:
        recommendations_df = recommend_for_cart(
            cart_product_ids=cart_product_ids,
            top_k=top_k,
            verbose=False
        )
        
        if recommendations_df.empty:
            return {
                'success': False,
                'error': 'No recommendations found',
                'recommendations': []
            }
        
        # Convert to JSON-serializable format
        recommendations = []
        for _, row in recommendations_df.iterrows():
            rec = {
                'product_id': row['product_id'],
                'score': float(row['final_score']),
                'visual_similarity': float(row['visual_similarity']),
                'co_purchase_score': float(row['co_purchase_score']),
            }
            
            # Add optional fields if available
            for field in ['product_name', 'price', 'collection', 'category']:
                if field in row and pd.notna(row[field]):
                    rec[field] = str(row[field])
            
            # Add image URL
            rec['image_url'] = f"/images/{row['product_id']}.jpg"
            
            recommendations.append(rec)
        
        return {
            'success': True,
            'cart_size': len(cart_product_ids),
            'recommendations_count': len(recommendations),
            'recommendations': recommendations,
            'metadata': {
                'model': 'cart_based_visual_similarity',
                'embedding_model': 'MobileNetV2',
                'weights': {
                    'visual': VISUAL_SIMILARITY_WEIGHT,
                    'co_purchase': CO_PURCHASE_WEIGHT
                }
            }
        }
        
    except Exception as e:
        return {
            'success': False,
            'error': str(e),
            'recommendations': []
        }

# Test API function
print("🚀 Testing production API function...\n")
api_result = get_cart_recommendations_api(sample_cart, top_k=5)

print("API Response:")
print(json.dumps(api_result, indent=2)[:1000] + "..." if len(json.dumps(api_result, indent=2)) > 1000 else json.dumps(api_result, indent=2))

## 🎯 Summary & Next Steps

### ✅ What This Notebook Does
1. **Loads pre-computed embeddings** from mounted image directory
2. **Extracts new embeddings on-demand** for products not in cache
3. **Builds co-purchase patterns** from GA4 interaction data
4. **Generates recommendations** based on cart contents using visual similarity + co-purchase signals
5. **Applies business rules** (same collection boost, price range filtering)
6. **Provides production API** with JSON output

### 🔧 Integration Steps
1. **Save embeddings** regularly (run cell 3 to extract for all products)
2. **Update co-purchase matrix** daily/weekly from new interaction data
3. **Call API function** from your e-commerce backend:
   ```python
   result = get_cart_recommendations_api(cart_product_ids=['123', '456'], top_k=10)
   ```
4. **Display recommendations** in UI with images from mounted directory

### 📈 Performance Optimization
- Pre-compute all embeddings to avoid on-demand extraction
- Cache recommendations for common cart combinations
- Use approximate nearest neighbors (FAISS) for large catalogs
- A/B test different weight combinations

### 🎨 Customization Options
- Adjust `VISUAL_SIMILARITY_WEIGHT` and `CO_PURCHASE_WEIGHT`
- Add more filters: brand preference, material, style tags
- Implement "Complete the Look" logic for jewelry sets
- Add trending/seasonal boost for certain products